## Step 1. Imports

In [0]:
from pyspark.sql import functions as F

from notebooks._shared.configuration import AppConfig
from notebooks._shared.contracts import BRONZE_MOVIES, BRONZE_CREDITS

## Step 2. Configuração

Resolve os namespaces Bronze e Silver a partir da configuração fornecida pelo ambiente.

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("silver_schema", "")

In [0]:
config = AppConfig(
    catalog=dbutils.widgets.get("catalog"),
    bronze_schema=dbutils.widgets.get("bronze_schema"),
    silver_schema=dbutils.widgets.get("silver_schema"),
)

movies_table = f"{config.bronze_namespace}.{BRONZE_MOVIES.name}"
credits_table = f"{config.bronze_namespace}.{BRONZE_CREDITS.name}"

## Step 3. Leitura Bronze

lê os datasets Bronze que fornecem os dados de filmes e créditos para a transformação Silver

In [0]:
movies_bronze_df = spark.table(movies_table)
credits_bronze_df = spark.table(credits_table)

## Step 4. Parsing dos campos semiestruturados

converte os campos semiestruturados da Bronze para estruturas Spark conforme os formatos validados na investigação Silver.

In [0]:
movies_nested_schemas = {
    "genres": "ARRAY<STRUCT<id: BIGINT, name: STRING>>",
    "keywords": "ARRAY<STRUCT<id: BIGINT, name: STRING>>",
    "production_companies": "ARRAY<STRUCT<id: BIGINT, name: STRING>>",
    "production_countries": "ARRAY<STRUCT<iso_3166_1: STRING, name: STRING>>",
    "spoken_languages": "ARRAY<STRUCT<iso_639_1: STRING, name: STRING>>",
}

credits_nested_schemas = {
    "cast": (
        "ARRAY<STRUCT<cast_id: BIGINT, character: STRING, credit_id: STRING, "
        "gender: BIGINT, id: BIGINT, name: STRING, order: BIGINT>>"
    ),
    "crew": (
        "ARRAY<STRUCT<credit_id: STRING, department: STRING, gender: BIGINT, "
        "id: BIGINT, job: STRING, name: STRING>>"
    ),
}

movies_parsed_df = movies_bronze_df.select(
    "*",
    *[
        F.from_json(F.col(column), schema).alias(f"{column}_parsed")
        for column, schema in movies_nested_schemas.items()
    ],
)

credits_parsed_df = credits_bronze_df.select(
    "*",
    *[
        F.from_json(F.col(column), schema).alias(f"{column}_parsed")
        for column, schema in credits_nested_schemas.items()
    ],
)

## Step 5. Transformações Silver

produz as oito entidades Silver aplicando a tipagem e as normalizações validadas na investigação

### 5.1 movie

In [0]:
movie_df = movies_parsed_df.select(
    F.col("id").cast("bigint").alias("movie_id"),
    F.col("budget").cast("bigint").alias("budget"),
    F.col("homepage").cast("string").alias("homepage"),
    F.col("original_language").cast("string").alias("original_language"),
    F.col("original_title").cast("string").alias("original_title"),
    F.col("overview").cast("string").alias("overview"),
    F.col("popularity").cast("double").alias("popularity"),
    F.to_date("release_date").alias("release_date"),
    F.col("revenue").cast("bigint").alias("revenue"),
    F.col("runtime").cast("double").alias("runtime"),
    F.col("status").cast("string").alias("status"),
    F.col("tagline").cast("string").alias("tagline"),
    F.col("title").cast("string").alias("title"),
    F.col("vote_average").cast("double").alias("vote_average"),
    F.col("vote_count").cast("bigint").alias("vote_count"),
    "_ingestion_id",
)

### 5.2 movie_genre

In [0]:
movie_genre_df = movies_parsed_df.select(
    F.col("id").cast("bigint").alias("movie_id"),
    F.explode("genres_parsed").alias("genre"),
    "_ingestion_id",
).select(
    "movie_id",
    F.col("genre.id").alias("genre_id"),
    F.col("genre.name").alias("genre_name"),
    "_ingestion_id",
)

### 5.3 movie_keyword

In [0]:
movie_keyword_df = movies_parsed_df.select(
    F.col("id").cast("bigint").alias("movie_id"),
    F.explode("keywords_parsed").alias("keyword"),
    "_ingestion_id",
).select(
    "movie_id",
    F.col("keyword.id").alias("keyword_id"),
    F.col("keyword.name").alias("keyword_name"),
    "_ingestion_id",
)

### 5.4 movie_production_company

In [0]:
movie_production_company_df = movies_parsed_df.select(
    F.col("id").cast("bigint").alias("movie_id"),
    F.explode("production_companies_parsed").alias("production_company"),
    "_ingestion_id",
).select(
    "movie_id",
    F.col("production_company.id").alias("company_id"),
    F.col("production_company.name").alias("company_name"),
    "_ingestion_id",
)

### 5.5 movie_production_country

In [0]:
movie_production_country_df = movies_parsed_df.select(
    F.col("id").cast("bigint").alias("movie_id"),
    F.explode("production_countries_parsed").alias("production_country"),
    "_ingestion_id",
).select(
    "movie_id",
    F.col("production_country.iso_3166_1").alias("country_code"),
    F.col("production_country.name").alias("country_name"),
    "_ingestion_id",
)

### 5.6 movie_spoken_language

In [0]:
movie_spoken_language_df = movies_parsed_df.select(
    F.col("id").cast("bigint").alias("movie_id"),
    F.explode("spoken_languages_parsed").alias("spoken_language"),
    "_ingestion_id",
).select(
    "movie_id",
    F.col("spoken_language.iso_639_1").alias("language_code"),
    F.col("spoken_language.name").alias("language_name"),
    "_ingestion_id",
)

### 5.7 cast_credit

In [0]:
cast_credit_df = credits_parsed_df.select(
    F.col("movie_id").cast("bigint").alias("movie_id"),
    F.explode("cast_parsed").alias("cast_member"),
    "_ingestion_id",
).select(
    "movie_id",
    F.col("cast_member.credit_id").alias("credit_id"),
    F.col("cast_member.id").alias("person_id"),
    F.col("cast_member.name").alias("person_name"),
    F.col("cast_member.cast_id").alias("cast_id"),
    F.col("cast_member.character").alias("character"),
    F.col("cast_member.gender").alias("gender"),
    F.col("cast_member.order").alias("cast_order"),
    "_ingestion_id",
)

### 5.8 crew_credit

In [0]:
crew_credit_df = credits_parsed_df.select(
    F.col("movie_id").cast("bigint").alias("movie_id"),
    F.explode("crew_parsed").alias("crew_member"),
    "_ingestion_id",
).select(
    "movie_id",
    F.col("crew_member.credit_id").alias("credit_id"),
    F.col("crew_member.id").alias("person_id"),
    F.col("crew_member.name").alias("person_name"),
    F.col("crew_member.gender").alias("gender"),
    F.col("crew_member.department").alias("department"),
    F.col("crew_member.job").alias("job"),
    "_ingestion_id",
)

## Step 6. Materialização Silver

materializa as oito entidades no namespace Silver. A reexecução integral com `overwrite` após falha parcial foi validada previamente na investigação (silver_modeling_validation step 9)

In [0]:
silver_dataframes = {
    "movie": movie_df,
    "movie_genre": movie_genre_df,
    "movie_keyword": movie_keyword_df,
    "movie_production_company": movie_production_company_df,
    "movie_production_country": movie_production_country_df,
    "movie_spoken_language": movie_spoken_language_df,
    "cast_credit": cast_credit_df,
    "crew_credit": crew_credit_df,
}

for entity, dataframe in silver_dataframes.items():
    table = f"{config.silver_namespace}.{entity}"

    try:
        (
            dataframe.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(table)
        )

        print(f"OK: '{table}' materializada.")

    except Exception:
        print(f"ERRO: falha ao materializar '{table}'.")
        raise

## Step 7. Validação da materialização

Confirma que as entidades produzidas pelo processamento foram materializadas no namespace Silver com a cardinalidade esperada para esta execução.

In [0]:
for entity, dataframe in silver_dataframes.items():
    table = f"{config.silver_namespace}.{entity}"

    expected_count = dataframe.count()
    persisted_count = spark.table(table).count()

    if persisted_count != expected_count:
        raise RuntimeError(
            f"Validação falhou para '{table}': "
            f"esperado={expected_count}, persistido={persisted_count}."
        )

    print(
        f"OK: '{table}' validada "
        f"(esperado={expected_count}, persistido={persisted_count})."
    )

## Step 8. Finalização

In [0]:
print("Silver transformation completed successfully")
print(f"Bronze Namespace: {config.bronze_namespace}")
print(f"Silver Namespace: {config.silver_namespace}")
print(f"Entities Materialized: {len(silver_dataframes)}")